# 面试问题：知识蒸馏怎样设计，Temperature、KL 与 `T²` 分别做什么？

**一句话回答**：冻结且处于 eval 模式的 teacher 产生软分布；student 同时优化硬标签交叉熵与 `T` 温度下的 teacher→student KL。高温暴露类别间相似度，`T²` 抵消 softmax 梯度随温度缩小的尺度。需要再定义 feature/logit 对齐、padding mask、teacher cache 版本和离线/线上指标。

本 Notebook 用 PyTorch 基础算子手写 KL、mask、feature adapter 与训练循环，不调用蒸馏框架。面试时还要主动区分“teacher 更准”与“student 值得部署”：前者只是知识来源，后者必须在固定延迟、显存和吞吐预算下与未蒸馏 student 公平比较，并检查长尾类别、置信度和数据漂移后的表现。

In [ ]:
import hashlib, json, math
import numpy as np
import torch
from torch import nn

SEED100=10001; torch.manual_seed(SEED100)
x100=torch.randn(240,5); teacher_w100=torch.tensor([[2.,-1.,.2],[-1.,2.,.3],[.5,.2,-1.5],[1.,1.,-.5],[-.4,.3,1.8]])
teacher_logits100=x100@teacher_w100; y100=teacher_logits100.argmax(-1)
assert x100.shape==(240,5) and teacher_logits100.shape==(240,3)
assert set(y100.tolist())=={0,1,2}
assert SEED100==10001

## 1. 温度软化分布，保留“暗知识”

`softmax(z/T)` 中 `T>1` 拉平分布，让非真类之间的相对关系可见。teacher 认为“猫比卡车更像狗”的信息不存在于 one-hot 标签里。训练时使用同一 T 比较 teacher/student；推理仍用 `T=1`。

In [ ]:
def log_softmax100(z,T=1.):
    if T<=0: raise ValueError("temperature_contract")
    u=z/T; return u-torch.logsumexp(u,dim=-1,keepdim=True)
def entropy100(p): return -(p*torch.log(p.clamp_min(1e-30))).sum(-1)
probe100=torch.tensor([[8.,2.,-3.]]); p1_100=log_softmax100(probe100,1).exp(); p4_100=log_softmax100(probe100,4).exp()
assert torch.allclose(p1_100.sum(-1),torch.ones(1)) and torch.allclose(p4_100.sum(-1),torch.ones(1))
assert entropy100(p4_100)>entropy100(p1_100)
try: log_softmax100(probe100,0); raise AssertionError("bad T accepted")
except ValueError as e: assert str(e)=="temperature_contract"

## 2. KL 方向与 `T²` 缩放

目标是 `KL(q_teacher || p_student)=Σq(logq-logp)`，teacher 是固定目标。softmax 对 logits 的导数含 `1/T`，KL 的梯度还会再缩小一次，因此乘 `T²` 让不同温度下梯度量级更可比。它不改变同一 T 下的最优点。

In [ ]:
def kd_kl100(student,teacher,T=2.,mask=None):
    logp=log_softmax100(student,T); logq=log_softmax100(teacher,T); per=(logq.exp()*(logq-logp)).sum(-1)*(T*T)
    if mask is None: return per.mean()
    mask=mask.to(per.dtype); return (per*mask).sum()/mask.sum().clamp_min(1)
s_probe100=torch.tensor([[1.,0.,-1.]],requires_grad=True); t_probe100=torch.tensor([[2.,-.5,-1.]])
kl100=kd_kl100(s_probe100,t_probe100,3); kl100.backward()
assert kl100>=-1e-6 and torch.isfinite(s_probe100.grad).all()
assert abs(float(s_probe100.grad.sum()))<1e-6
assert abs(float(kd_kl100(t_probe100,t_probe100,3)))<1e-6

## 3. 硬标签与软目标各自负责什么

常用总损失是 `α·T²KL +(1-α)·CE`。硬标签锚定任务真值，软目标传递 teacher 的类间结构。α 与 T 必须联合调参；若 teacher 在某 slice 明显有偏差，可降低该样本蒸馏权重，而不是盲目模仿。

In [ ]:
def hard_ce100(logits,target): return (-log_softmax100(logits,1)[torch.arange(len(target)),target]).mean()
student_probe100=torch.randn(12,3,requires_grad=True); teacher_probe100=torch.randn(12,3); labels_probe100=torch.arange(12)%3; alpha100=.7
total100=alpha100*kd_kl100(student_probe100,teacher_probe100,3)+(1-alpha100)*hard_ce100(student_probe100,labels_probe100); total100.backward()
assert total100>0 and torch.isfinite(total100)
assert student_probe100.grad is not None and torch.isfinite(student_probe100.grad).all()
assert teacher_probe100.grad is None

## 4. Teacher 必须冻结并切换到 eval

`no_grad` 只关闭 autograd，不会自动关闭 Dropout/更新 BatchNorm；`eval` 也不会自动冻结参数。两者都要做。离线 cache teacher logits 时还需记录 teacher checkpoint、tokenizer、温度前的原始 logits、样本 ID 与 mask。

In [ ]:
class Teacher100(nn.Module):
    def __init__(self): super().__init__(); self.drop=nn.Dropout(.8); self.register_buffer("weight",teacher_w100.clone())
    def forward(self,x): return self.drop(x)@self.weight
teacher100=Teacher100(); teacher100.eval()
with torch.no_grad(): out_a100=teacher100(x100[:8]); out_b100=teacher100(x100[:8])
assert torch.equal(out_a100,out_b100)
assert not out_a100.requires_grad and not any(p.requires_grad for p in teacher100.parameters())
assert torch.equal(out_a100,x100[:8]@teacher_w100)

## 5. Feature distillation 需要显式 adapter

teacher/student 隐层维度不同，不能直接相减；用可训练投影把 student feature 映射到 teacher 空间，再做 MSE 或 cosine loss。层对应、归一化位置和 feature 权重均是设计选择，过强会限制小模型形成自己的表示。

In [ ]:
student_feat100=torch.randn(16,4,requires_grad=True); teacher_feat100=torch.randn(16,7); adapter100=nn.Parameter(torch.randn(4,7)*.1)
projected100=student_feat100@adapter100; feature_loss100=((projected100-teacher_feat100)**2).mean(); feature_loss100.backward()
assert projected100.shape==teacher_feat100.shape
assert student_feat100.grad is not None and adapter100.grad is not None
assert torch.isfinite(adapter100.grad).all() and feature_loss100>0

## 6. 序列蒸馏必须按有效 token 归一化

padding token 不应贡献 KL；不同长度样本也不应因为 padding 多而权重异常。计算逐 token KL，乘 attention mask 后除以全局有效 token 数。分布式训练则需要 AllReduce 分子与分母。

In [ ]:
ss100=torch.randn(2,4,3); tt100=torch.randn(2,4,3); mask100=torch.tensor([[1,1,1,0],[1,0,0,0]],dtype=torch.bool)
masked_kd100=kd_kl100(ss100,tt100,2,mask100); flat_valid100=mask100.reshape(-1); reference_kd100=kd_kl100(ss100.reshape(-1,3)[flat_valid100],tt100.reshape(-1,3)[flat_valid100],2)
assert torch.allclose(masked_kd100,reference_kd100,atol=1e-7)
assert int(mask100.sum())==4
assert torch.isfinite(masked_kd100) and masked_kd100>=-1e-6

## 7. Teacher cache 是有版本的数据产物

缓存 logits 能省 teacher 推理，但会增加存储与版本耦合。FP16 缓存通常足够，必要时只存 top-k logits 并保留剩余质量的近似；任何 tokenizer、类别顺序或 teacher 权重变化都必须失效重算。

In [ ]:
raw100=teacher_logits100[:32]; cached100=raw100.half(); p_raw100=log_softmax100(raw100,3).exp(); p_cache100=log_softmax100(cached100.float(),3).exp()
cache_meta100={"teacher":"linear-v1","dtype":"float16","shape":list(cached100.shape),"class_order":[0,1,2]}; cache_digest100=hashlib.sha256(json.dumps(cache_meta100,sort_keys=True).encode()).hexdigest()
assert cached100.element_size()==2 and raw100.element_size()==4
assert torch.max(torch.abs(p_raw100-p_cache100))<5e-4
assert len(cache_digest100)==64 and cache_meta100["shape"]==[32,3]

## 8. 端到端 student 训练与验收

下面手写一个线性 student 与更新循环。真实项目要同时比较 student-only baseline、teacher 上限、延迟/显存、总体与关键 slice 指标及校准；蒸馏成功不是 loss 下降，而是在相同部署预算下优于基线。

In [ ]:
class Student100(nn.Module):
    def __init__(self): super().__init__(); self.weight=nn.Parameter(torch.zeros(5,3)); self.bias=nn.Parameter(torch.zeros(3))
    def forward(self,x): return x@self.weight+self.bias
student100=Student100(); losses100=[]
for _ in range(100):
    slog=student100(x100); loss=.75*kd_kl100(slog,teacher_logits100,3)+.25*hard_ce100(slog,y100); losses100.append(float(loss.detach())); loss.backward()
    with torch.no_grad():
        for p in student100.parameters(): p-=.12*p.grad; p.grad=None
acc100=float((student100(x100).argmax(-1)==y100).float().mean()); manifest100={"teacher":"linear-v1","student":"linear-5x3","temperature":3.,"alpha":.75,"mask":"valid_token_mean"}; digest100=hashlib.sha256(json.dumps(manifest100,sort_keys=True).encode()).hexdigest()
assert losses100[-1]<losses100[0]*.15
assert acc100>.9 and all(math.isfinite(v) for v in losses100)
assert len(digest100)==64 and manifest100["alpha"]==.75

## 面试总结

推荐按 **teacher 冻结/eval → 温度软目标 → `KL(teacher||student)·T²` → 硬软损失 → feature adapter → token mask → cache 版本 → 同预算验收** 回答。蒸馏不是“复制输出”，而是在容量受限的 student 中选择要迁移的信息与代价。

延伸阅读：[Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)、[PyTorch KLDivLoss 语义](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html)、[DistilBERT](https://arxiv.org/abs/1910.01108)。